<a href="https://colab.research.google.com/github/banteamlak1888/Floridan-University-Fundamentals-AI-Trianing/blob/main/Project%201%5B%5DCostumer%20Chunk%20Predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
"""
Customer Churn Prediction - Module 5 Implementation Script
Covers Tasks 1 to 8: Problem Refinement, Solution Design, Data Prep, EDA,
Baseline Modeling, Model Improvement, Evaluation, and System Testing.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

def main():
    print("=== TASK 1 & 2: PROBLEM REFINEMENT & AI SOLUTION DESIGN ===")
    print("Project: Customer Churn Prediction")
    print("Objective: Predict whether a customer will churn (binary classification) to enable proactive retention.")

    # Task 3: Collect and Prepare Data
    print("\n=== TASK 3: DATA PREPARATION ===")
    train_path = "customer_churn_dataset-training-master.csv"
    test_path = "customer_churn_dataset-testing-master.csv"

    if not os.path.exists(train_path) or not os.path.exists(test_path):
        print("Dataset files not found. Please ensure training and testing CSV files are in the directory.")
        return

    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    print(f"Training set shape: {df_train.shape}")
    print(f"Testing set shape: {df_test.shape}")

    # Clean column names (strip whitespace if any)
    df_train.columns = df_train.columns.str.strip()
    df_test.columns = df_test.columns.str.strip()

    # Identify target and features
    # Assuming standard churn dataset structure; adapt column names if necessary based on CSV headers
    target_col = 'Churn' if 'Churn' in df_train.columns else df_train.columns[-1]
    print(f"Target variable identified: {target_col}")

    # Handle missing values & basic cleaning
    df_train = df_train.dropna(subset=[target_col])

    X_train = df_train.drop(columns=[target_col])
    y_train = df_train[target_col]
    X_test = df_test.drop(columns=[target_col], errors='ignore')
    y_test = df_test[target_col] if target_col in df_test.columns else None

    # Task 4: Exploratory Data Analysis (EDA) Summary
    print("\n=== TASK 4: EXPLORATORY DATA ANALYSIS (EDA) ===")
    print("Target class distribution:")
    print(y_train.value_counts(normalize=True))

    # Task 5: Build a Baseline Model
    print("\n=== TASK 5: BUILD A BASELINE MODEL ===")
    # Identify numerical and categorical columns
    numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    baseline_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ])

    baseline_model.fit(X_train, y_train)

    # Internal validation/training check
    y_train_pred = baseline_model.predict(X_train)
    print("Baseline Training Accuracy:", accuracy_score(y_train, y_train_pred))

    # Task 6: Improve the Model
    print("\n=== TASK 6: IMPROVE THE MODEL (RANDOM FOREST & TUNING) ===")
    improved_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10))
    ])

    improved_model.fit(X_train, y_train)
    y_train_imp_pred = improved_model.predict(X_train)
    print("Improved Model Training Accuracy:", accuracy_score(y_train, y_train_imp_pred))

    # Task 7: Evaluate the Model (Using Test Set or Cross-Validation)
    print("\n=== TASK 7: EVALUATE THE MODEL ===")
    if y_test is not None and len(y_test) > 0:
        y_test_pred = improved_model.predict(X_test)
        print("\nTest Set Evaluation Metrics:")
        print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
        print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
        print(f"Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
        print(f"F1-Score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_test_pred))
    else:
        print("Performing 5-Fold Cross-Validation on Training Data as test set labels are separate/missing:")
        cv_scores = cross_val_score(improved_model, X_train, y_train, cv=5, scoring='f1')
        print(f"Cross-Validation F1 Scores: {cv_scores}")
        print(f"Mean CV F1 Score: {cv_scores.mean():.4f}")

    # Task 8: Test the Complete Solution (System & Edge Case Testing)
    print("\n=== TASK 8: TEST THE COMPLETE SOLUTION (EDGE CASES) ===")
    test_cases = [
        ("Normal Complete Input", X_train.iloc[0:1]),
        ("Missing Value Handling", X_train.iloc[0:1].assign(**{numeric_features[0]: np.nan} if numeric_features else {})),
    ]

    for case_name, sample_input in test_cases:
        try:
            pred = improved_model.predict(sample_input)
            print(f"Test Case [{case_name}]: PASSED (Prediction: {pred[0]})")
        except Exception as e:
            print(f"Test Case [{case_name}]: FAILED with error: {e}")

    print("\nImplementation pipeline executed successfully!")

if __name__ == "__main__":
    main()

=== TASK 1 & 2: PROBLEM REFINEMENT & AI SOLUTION DESIGN ===
Project: Customer Churn Prediction
Objective: Predict whether a customer will churn (binary classification) to enable proactive retention.

=== TASK 3: DATA PREPARATION ===
Training set shape: (440833, 12)
Testing set shape: (64374, 12)
Target variable identified: Churn

=== TASK 4: EXPLORATORY DATA ANALYSIS (EDA) ===
Target class distribution:
Churn
1.0    0.567107
0.0    0.432893
Name: proportion, dtype: float64

=== TASK 5: BUILD A BASELINE MODEL ===
Baseline Training Accuracy: 0.9806298090882695

=== TASK 6: IMPROVE THE MODEL (RANDOM FOREST & TUNING) ===
Improved Model Training Accuracy: 0.9959009327816493

=== TASK 7: EVALUATE THE MODEL ===

Test Set Evaluation Metrics:
Accuracy:  0.4904
Precision: 0.4818
Recall:    1.0000
F1-Score:  0.6502

Confusion Matrix:
[[ 1079 32802]
 [    1 30492]]

=== TASK 8: TEST THE COMPLETE SOLUTION (EDGE CASES) ===
Test Case [Normal Complete Input]: PASSED (Prediction: 1.0)
Test Case [Missin